# PyTorch recommendation-matching demo (learning notebook)

**What this is.** A personal PyTorch learning exercise, built on Curalina's
real supplier catalogue data (Celadon, Lazzoni, Luxus), that demonstrates
the actual shape of the recommendation problem: **given a user's quiz
answers (room type, style, atmosphere, budget), score every real product in
the catalogue for how well it matches, then assemble a set that fills the
room** — e.g. "Bedroom" + "Organic Modern" -> a bed, a nightstand, a dresser,
wall art, all in that style, within budget.

**What this is not.** This is *not* one of the project's formal evaluation
notebooks (`R01`-`R03`, `V01`-`V03`, `G01`-`G03`, `D01`) and does not follow
the seven-section notebook standard (`agentic_flow/16_notebook_standard.md`).
It carries no accuracy, feasibility, or acceptance claim, and does not feed
into or override `ADR-0006` (the real ranking service's leakage firewall) or
`ADR-0013` (the formal R03 no-go on real-furniture-data *bundle
composition*, which is why the actual `recommendation` service's real
`Product` domain object deliberately excludes `room_type`/`design_style`
from anything its ranking logic can read). This notebook uses those fields
freely because it is a separate, informal exercise in learning PyTorch on a
realistic dataset, not a candidate implementation for the live service.

## The actual task

There's no real "user clicked this, so it must be good" signal in this
data (no purchase or rating history), so this notebook trains a
**compatibility scorer**: for a `(query, product)` pair, predict whether the
product's own room type / style / atmosphere match the query's. The
positive examples are real (a product's own tags genuinely describe it);
the negative examples are query profiles we deliberately mismatch against
a product on purpose (a standard technique called negative sampling — not
fabricated data, just examples of *non*-matches to teach the model what
"doesn't fit" looks like). Once trained, we use the scorer at inference
time to rank *all* products against a brand-new query and assemble a set,
one item per needed category, within budget — the same shape as the quiz
-> recommendation flow in the real app.

## Running locally vs. in Google Colab

**Local (default):** `DATA_DIR` below points straight at the three supplier
workbooks on disk.

**Google Colab:** upload the `Supplier CSV Files` folder to Google Drive,
then before the data-loading cell:

```python
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/Supplier CSV Files')
```

Colab also needs one extra install cell (uncomment it below) — locally,
`pip install torch pandas openpyxl matplotlib` once, outside the notebook,
is enough.


In [ ]:
# Uncomment in Colab (locally, install these once from your terminal instead):
# !pip install torch pandas openpyxl matplotlib

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# --- Path configuration --------------------------------------------------
# Local default. In Colab, mount Drive first (see markdown above) and
# reassign this to the Drive path instead.
DATA_DIR = Path(os.environ.get(
    "CURALINA_SUPPLIER_DATA_DIR",
    "/Users/rjsalmon/Documents/Humber/misc.curalina/Supplier CSV Files",
))

WORKBOOKS = {
    "Celadon": DATA_DIR / "Celadon CSV Programmer Handoff.xlsx",
    "Lazzoni": DATA_DIR / "Lazzoni CSV Programmer Handoff.xlsx",
    "Luxus": DATA_DIR / "Luxus Programmer Handoff.xlsx",
}
for supplier, path in WORKBOOKS.items():
    print(supplier, "->", path, "exists:", path.exists())


## 1. Load and clean the three supplier workbooks

These are named "CSV" files but are actually `.xlsx`, and **the three
suppliers don't share column names or column order** (confirmed by
inspecting the headers directly — e.g. Celadon calls its key column `SKU`,
Luxus calls the same idea `SUPPLIER SKU` and leaves it blank for most
rows). Celadon's sheet is artwork only and has no `Product Type` column at
all — every Celadon row gets `category = "Wall Art"` directly, since that's
what its own sheet name says it is.


In [ ]:
# Canonical column name -> the possible header spellings we've actually seen.
COLUMN_ALIASES = {
    "sku": ["SKU", "Supplier SKU", "SUPPLIER SKU"],
    "product_name": ["Product Name"],
    "supplier": ["Supplier"],
    "category": ["Product Type"],
    "trade_price": ["Trade Price"],
    "retail_price": ["Retail Price"],
    "room_type": ["Room Type"],
    "design_style": ["Design Style"],
    "atmosphere": ["Atmosphere"],
    "width_in": ["Width (in)"],
    "depth_in": ["Depth (in)"],
    "height_in": ["Height (in)"],
    "overview": ["Product Overview"],
}


def load_and_rename(path: Path, supplier_label: str) -> pd.DataFrame:
    df = pd.read_excel(path, engine="openpyxl")
    rename_map = {}
    for canonical, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            if alias in df.columns:
                rename_map[alias] = canonical
                break
    df = df.rename(columns=rename_map)
    for canonical in COLUMN_ALIASES:
        if canonical not in df.columns:
            df[canonical] = np.nan
    df = df[list(COLUMN_ALIASES.keys())].copy()
    df["supplier"] = supplier_label  # trust the filename, not the sheet
    if supplier_label == "Celadon":
        df["category"] = "Wall Art"  # Celadon's ARTWORK sheet has no Product Type column
    return df


raw_frames = [load_and_rename(path, supplier) for supplier, path in WORKBOOKS.items()]
catalogue = pd.concat(raw_frames, ignore_index=True)
print(f"Loaded {len(catalogue)} rows across {len(WORKBOOKS)} suppliers.")
catalogue.head()


### Cleaning steps, explained

1. **Missing SKUs (Luxus).** Many Luxus rows have a blank SKU — fall back
   to the row index as a stable id rather than dropping the row.
2. **`retail_price == 0` is missing, not free.** A few rows (mostly Luxus)
   have `retail_price = 0` with no `trade_price` either — a data-entry
   placeholder, treated as missing for both price columns.
3. **`room_type`, `design_style`, and `atmosphere` are all multi-valued**
   (e.g. `"Mid-Century Scandinavian; Contemporary Luxe"` for a piece that
   fits two styles). We only take the *first* listed tag for each — good
   enough for this demo, not something you'd ship.
4. **Rows with no price or no category get dropped** — both are required
   for the matching task below (price for the budget step, category so we
   know what "one item per category" means).


In [ ]:
def first_tag(value):
    if not isinstance(value, str):
        return "unknown"
    return value.split(";")[0].strip() or "unknown"


clean = catalogue.copy()
clean["sku"] = clean["sku"].astype("object")
clean.loc[clean["sku"].isna(), "sku"] = [
    f"row_{i}" for i in clean.index[clean["sku"].isna()]
]

for price_col in ["trade_price", "retail_price"]:
    clean[price_col] = pd.to_numeric(clean[price_col], errors="coerce")
    clean.loc[clean[price_col] == 0, price_col] = np.nan

for tag_col in ["room_type", "design_style", "atmosphere"]:
    clean[tag_col] = clean[tag_col].apply(first_tag)

for dim_col in ["width_in", "depth_in", "height_in"]:
    clean[dim_col] = pd.to_numeric(clean[dim_col], errors="coerce")
    clean[dim_col] = clean[dim_col].fillna(clean[dim_col].median())

clean = clean.dropna(subset=["retail_price", "category"]).reset_index(drop=True)
print(f"{len(clean)} products left after cleaning.")
clean[["sku", "supplier", "category", "room_type", "design_style", "atmosphere", "retail_price"]].sample(8, random_state=1)


## 2. Build the matching dataset: (query, product, match) triples

For every real product, its own `(room_type, design_style, atmosphere)` is
a genuine positive query — a query built from a product's own tags should
score as a strong match for that product. For each positive, we generate a
few negatives by taking the *same product* and pairing it with a query
whose room type, style, or atmosphere we deliberately swap for a different
value — a query that shouldn't match it well. `NEGATIVES_PER_POSITIVE`
controls how many of those we generate per real product.

This is the standard "negative sampling" technique used to train
retrieval/matching models when you only have positive examples on hand —
we are not inventing fake products or fake prices, only pairing real
products with query profiles that don't describe them, so the model has
something to contrast against.


In [ ]:
NEGATIVES_PER_POSITIVE = 3

room_types = sorted(clean["room_type"].unique())
design_styles = sorted(clean["design_style"].unique())
atmospheres = sorted(clean["atmosphere"].unique())

rng = np.random.default_rng(42)


def build_pairs(products: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, product in products.iterrows():
        # Positive: the query is exactly this product's own tags.
        rows.append({
            "query_room_type": product["room_type"],
            "query_design_style": product["design_style"],
            "query_atmosphere": product["atmosphere"],
            "product_sku": product["sku"],
            "match": 1,
        })
        # Negatives: mismatch at least one of the three tags on purpose.
        for _ in range(NEGATIVES_PER_POSITIVE):
            query_room_type = rng.choice(room_types)
            query_style = rng.choice(design_styles)
            query_atmosphere = rng.choice(atmospheres)
            same_as_positive = (
                query_room_type == product["room_type"]
                and query_style == product["design_style"]
                and query_atmosphere == product["atmosphere"]
            )
            if same_as_positive:
                continue  # accidentally drew the positive combo again -- skip it
            rows.append({
                "query_room_type": query_room_type,
                "query_design_style": query_style,
                "query_atmosphere": query_atmosphere,
                "product_sku": product["sku"],
                "match": 0,
            })
    return pd.DataFrame(rows)


pairs = build_pairs(clean)
pairs = pairs.merge(
    clean[["sku", "category", "supplier", "retail_price", "width_in", "depth_in", "height_in"]],
    left_on="product_sku", right_on="sku", how="left",
)
print(f"{len(pairs)} (query, product) pairs -- {pairs['match'].mean():.1%} positive")
pairs.head()


## 3. Embeddings, explained

Our categorical columns are strings — a neural net needs numbers.
`nn.Embedding` learns a small dense vector *per category value* during
training (e.g. every `design_style` becomes an 8-number vector) instead of
one-hot encoding treating every category as equally different from every
other. Mechanically it's a lookup table: `nn.Embedding(num_categories,
embedding_dim)` stores one trainable vector per category, and indexing it
with a category's integer id returns that vector.

We need **two sets** of embedding tables here, because a query's room type
and a product's room type are conceptually different things even though
they share a vocabulary (a bedroom *query* and a bedroom *product tag* are
both "Bedroom", but the model shouldn't be forced to represent them
identically) — plus a couple of embeddings that only exist on the product
side (`category`, `supplier`).


In [ ]:
QUERY_VOCABS = {
    "query_room_type": {v: i for i, v in enumerate(room_types)},
    "query_design_style": {v: i for i, v in enumerate(design_styles)},
    "query_atmosphere": {v: i for i, v in enumerate(atmospheres)},
}
PRODUCT_CATEGORICAL_VOCABS = {
    "category": {v: i for i, v in enumerate(sorted(clean["category"].unique()))},
    "supplier": {v: i for i, v in enumerate(sorted(clean["supplier"].unique()))},
}
CONTINUOUS_COLUMNS = ["retail_price", "width_in", "depth_in", "height_in"]

for col, vocab in QUERY_VOCABS.items():
    pairs[f"{col}_id"] = pairs[col].map(vocab)
for col, vocab in PRODUCT_CATEGORICAL_VOCABS.items():
    pairs[f"{col}_id"] = pairs[col].map(vocab)

print({col: len(v) for col, v in {**QUERY_VOCABS, **PRODUCT_CATEGORICAL_VOCABS}.items()})


## 4. Tensors, train/test split, `DataLoader`

Same shape as your other notebooks. Three input tensors per example this
time (query categorical ids, product categorical ids, product continuous
features) plus the binary target, but `TensorDataset` and `DataLoader`
handle that exactly the same way as a two-tensor dataset.


In [ ]:
torch.manual_seed(42)

query_cols = list(QUERY_VOCABS.keys())
product_cat_cols = list(PRODUCT_CATEGORICAL_VOCABS.keys())

X_query = torch.tensor(pairs[[f"{c}_id" for c in query_cols]].values, dtype=torch.long)
X_product_cat = torch.tensor(pairs[[f"{c}_id" for c in product_cat_cols]].values, dtype=torch.long)
X_continuous = torch.tensor(pairs[CONTINUOUS_COLUMNS].values, dtype=torch.float32)
y = torch.tensor(pairs[["match"]].values, dtype=torch.float32)

# Standardize the continuous features -- price and dimensions live on very
# different scales, and an MLP trains far more reliably on roughly
# unit-scale numbers.
cont_mean, cont_std = X_continuous.mean(dim=0), X_continuous.std(dim=0).clamp(min=1e-6)
X_continuous = (X_continuous - cont_mean) / cont_std

n = len(pairs)
n_train = int(n * 0.8)
perm = torch.randperm(n)
train_idx, test_idx = perm[:n_train], perm[n_train:]

train_dataset = TensorDataset(X_query[train_idx], X_product_cat[train_idx], X_continuous[train_idx], y[train_idx])
test_dataset = TensorDataset(X_query[test_idx], X_product_cat[test_idx], X_continuous[test_idx], y[test_idx])
train_dl = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dl = DataLoader(test_dataset, batch_size=32, shuffle=True)

print(f"train: {len(train_dataset)} pairs, test: {len(test_dataset)} pairs")


## 5. The model

Same two-layer MLP shape as `SimpleLinearRegression` (`fc1` -> `relu` ->
`fc2`), with one embedding table per categorical column feeding into
`fc1` alongside the continuous features. The output is a single raw score
(a *logit*) — `nn.BCEWithLogitsLoss` applies the sigmoid internally during
training for numerical stability, and `predict_proba` applies it
explicitly to turn the score into a 0-1 match probability at inference
time.


In [ ]:
class RecommendationMatchModel(nn.Module):
    def __init__(self, query_vocab_sizes: dict[str, int], product_vocab_sizes: dict[str, int],
                 n_continuous: int, embedding_dim: int = 8, hidden: int = 32):
        super().__init__()
        self.query_embeddings = nn.ModuleDict({
            col: nn.Embedding(size, embedding_dim) for col, size in query_vocab_sizes.items()
        })
        self.product_embeddings = nn.ModuleDict({
            col: nn.Embedding(size, embedding_dim) for col, size in product_vocab_sizes.items()
        })
        total_in = embedding_dim * (len(query_vocab_sizes) + len(product_vocab_sizes)) + n_continuous
        self.fc1 = nn.Linear(total_in, hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, 1)

    def forward(self, x_query, x_product_cat, x_continuous):
        query_parts = [self.query_embeddings[col](x_query[:, i]) for i, col in enumerate(query_cols)]
        product_parts = [self.product_embeddings[col](x_product_cat[:, i]) for i, col in enumerate(product_cat_cols)]
        x = torch.cat([*query_parts, *product_parts, x_continuous], dim=1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x  # raw logit -- use BCEWithLogitsLoss, not BCELoss, on this directly

    def predict_proba(self, x_query, x_product_cat, x_continuous):
        with torch.no_grad():
            return torch.sigmoid(self.forward(x_query, x_product_cat, x_continuous))


device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

query_vocab_sizes = {col: len(QUERY_VOCABS[col]) for col in query_cols}
product_vocab_sizes = {col: len(PRODUCT_CATEGORICAL_VOCABS[col]) for col in product_cat_cols}
model = RecommendationMatchModel(query_vocab_sizes, product_vocab_sizes, n_continuous=len(CONTINUOUS_COLUMNS)).to(device)
model


## 6. Training loop

Same shape you've been using: one `losses` list per epoch, Adam, a
`device` move per batch. Because this is a classification problem (match /
no-match) rather than regression, the loss is `BCEWithLogitsLoss` instead
of `MSELoss`, and we also track accuracy each epoch — a loss number alone
doesn't tell you "is it actually getting the right products".


In [ ]:
epochs = 60
train_losses = []
test_losses = []
test_accuracies = []

criterion = nn.BCEWithLogitsLoss()
torch.manual_seed(42)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for batch_q, batch_pc, batch_cont, batch_y in train_dl:
        batch_q, batch_pc, batch_cont, batch_y = (
            batch_q.to(device), batch_pc.to(device), batch_cont.to(device), batch_y.to(device)
        )
        optimizer.zero_grad()
        y_pred = model(batch_q, batch_pc, batch_cont)
        loss = criterion(y_pred, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_train_loss = epoch_loss / len(train_dl)
    train_losses.append(avg_train_loss)

    model.eval()
    with torch.no_grad():
        test_loss = 0.0
        correct = 0
        total = 0
        for batch_q, batch_pc, batch_cont, batch_y in test_dl:
            batch_q, batch_pc, batch_cont, batch_y = (
                batch_q.to(device), batch_pc.to(device), batch_cont.to(device), batch_y.to(device)
            )
            logits = model(batch_q, batch_pc, batch_cont)
            test_loss += criterion(logits, batch_y).item()
            predicted = (torch.sigmoid(logits) > 0.5).float()
            correct += (predicted == batch_y).sum().item()
            total += batch_y.numel()
    avg_test_loss = test_loss / len(test_dl)
    test_losses.append(avg_test_loss)
    test_accuracies.append(correct / total)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch: {epoch+1}, Train Loss: {avg_train_loss:.4f}, "
              f"Test Loss: {avg_test_loss:.4f}, Test Accuracy: {test_accuracies[-1]:.1%}")


## 7. Plots

Academic-report style: train/test loss over epochs, and test accuracy over
epochs (does it plateau? overfit?).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(train_losses, label="train loss")
axes[0].plot(test_losses, label="test loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary cross-entropy loss")
axes[0].set_title("Training and test loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(test_accuracies)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Test accuracy")
axes[1].set_title("Match / no-match accuracy (held-out pairs)")
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()


## 8. The actual demo: recommend a full room from a query

This is the point of the whole exercise. Given a query like "Bedroom" +
"Organic Modern" + "Bright & Airy" + a budget, score **every real
product** in the catalogue, then greedily take the highest-scoring product
in each category the room needs, skipping any that would blow the budget.
This mirrors the real recommendation service's own bundle-assembly idea
(`LogicOnlyBundleComposer._select_lowest_price_per_category`) — pick one
item per required category — except the *selection* signal here is the
model's learned match score instead of lowest price.


In [ ]:
BEDROOM_CATEGORIES = ["Bed", "Nightstand", "Dresser", "Wall Art", "Bench"]


def recommend_room(query_room_type: str, query_style: str, query_atmosphere: str,
                    categories_needed: list[str], budget: float) -> pd.DataFrame:
    candidates = clean[clean["category"].isin(categories_needed)].reset_index(drop=True)

    q_room = torch.full((len(candidates),), QUERY_VOCABS["query_room_type"][query_room_type], dtype=torch.long)
    q_style = torch.full((len(candidates),), QUERY_VOCABS["query_design_style"][query_style], dtype=torch.long)
    q_atmo = torch.full((len(candidates),), QUERY_VOCABS["query_atmosphere"][query_atmosphere], dtype=torch.long)
    x_query = torch.stack([q_room, q_style, q_atmo], dim=1).to(device)

    cat_ids = candidates["category"].map(PRODUCT_CATEGORICAL_VOCABS["category"]).values
    sup_ids = candidates["supplier"].map(PRODUCT_CATEGORICAL_VOCABS["supplier"]).values
    x_product_cat = torch.tensor(np.stack([cat_ids, sup_ids], axis=1), dtype=torch.long).to(device)

    x_continuous = torch.tensor(candidates[CONTINUOUS_COLUMNS].values, dtype=torch.float32)
    x_continuous = ((x_continuous - cont_mean) / cont_std).to(device)

    scores = model.predict_proba(x_query, x_product_cat, x_continuous).cpu().numpy().flatten()
    candidates = candidates.assign(match_score=scores).sort_values("match_score", ascending=False)

    chosen_rows = []
    remaining_budget = budget
    for category in categories_needed:
        options = candidates[candidates["category"] == category]
        affordable = options[options["retail_price"] <= remaining_budget]
        if affordable.empty:
            continue  # nothing in this category fits what's left of the budget
        pick = affordable.iloc[0]
        chosen_rows.append(pick)
        remaining_budget -= pick["retail_price"]

    return pd.DataFrame(chosen_rows)[["product_name", "supplier", "category", "match_score", "retail_price"]]


recommended = recommend_room(
    query_room_type="Bedroom",
    query_style="Organic Modern",
    query_atmosphere="Bright & Airy",
    categories_needed=BEDROOM_CATEGORIES,
    budget=5000,
)
total_spent = recommended["retail_price"].sum()
print(f"Recommended set (total ${total_spent:,.2f} of $5,000 budget):")
recommended


## What this does and doesn't show you

This demonstrates the actual shape of the recommendation problem — score
real products against a user's stated preferences, then assemble a set
that fills the room within budget — using real PyTorch mechanics
(dual-sided embeddings, `TensorDataset`/`DataLoader`, a binary-classification
training loop, loss/accuracy plotting). It is **not** the recommendation
service's real ranking pipeline: it doesn't use the service's actual
`Product`/`Profile`/`Bundle` domain objects, its negative-sampling scheme
is a teaching simplification rather than a reviewed evaluation design, and
the "does this even work well" question hasn't been checked by anyone
except this notebook's own accuracy number.

If an embedding-based ranking or recommendation model is wanted for the
actual service, that decision — architecture, evaluation methodology,
accept/reject — is `ai-ml-lead`'s call per this project's roster
(`AGENTS.md`), following the same evidence-notebook standard as `R01`-`R03`,
and it would need to respect `ADR-0006`'s leakage firewall (which this
notebook deliberately does not, since it isn't feeding the real service).
This notebook is a stepping stone toward being able to read and reason
about that work, not a substitute for it.
